# 03 : Implementation of Viterbi Decoding Algorithm for Part-of-Speech (POS) Tagging

# Step 1: Install & Import

In [2]:
import nltk
nltk.download('treebank')

from nltk.corpus import treebank

[nltk_data] Downloading package treebank to
[nltk_data]     C:\Users\kambl\AppData\Roaming\nltk_data...
[nltk_data]   Package treebank is already up-to-date!


# Step 2: Load Data

In [3]:
data = treebank.tagged_sents()

# Step 3: Create Counts

In [4]:
transition = {}
emission = {}
tag_count = {}

for sent in data:
    prev_tag = "START"
    
    for word, tag in sent:
        # Transition
        if prev_tag not in transition:
            transition[prev_tag] = {}
        transition[prev_tag][tag] = transition[prev_tag].get(tag, 0) + 1

        # Emission
        if tag not in emission:
            emission[tag] = {}
        emission[tag][word] = emission[tag].get(word, 0) + 1

        # Tag count
        tag_count[tag] = tag_count.get(tag, 0) + 1

        prev_tag = tag

# Step 4: Convert to Probability

In [5]:
# Transition probability
for prev in transition:
    total = sum(transition[prev].values())
    for tag in transition[prev]:
        transition[prev][tag] /= total

# Emission probability
for tag in emission:
    total = sum(emission[tag].values())
    for word in emission[tag]:
        emission[tag][word] /= total

# Step 5: Simple Viterbi Function

In [7]:
def viterbi(sentence, tags):
    V = [{}]
    path = {}

    # Start
    for tag in tags:
        V[0][tag] = transition["START"].get(tag, 0) * emission[tag].get(sentence[0], 1e-6)
        path[tag] = [tag]

    # Loop
    for t in range(1, len(sentence)):
        new_path = {}
        V.append({})

        for curr_tag in tags:
            max_prob = 0
            best_tag = None

            for prev_tag in tags:
                prob = V[t-1][prev_tag] * transition.get(prev_tag, {}).get(curr_tag, 0) * emission[curr_tag].get(sentence[t], 1e-6)

                if prob > max_prob:
                    max_prob = prob
                    best_tag = prev_tag

            V[t][curr_tag] = max_prob
            new_path[curr_tag] = path[best_tag] + [curr_tag]

        path = new_path

    # Final best tag
    best = max(V[-1], key=V[-1].get)
    return path[best]

# Step 6: Run Example

In [8]:
sentence = ["The", "dog", "barks"]
tags = list(tag_count.keys())

result = viterbi(sentence, tags)

for w, t in zip(sentence, result):
    print(w, "->", t)

The -> DT
dog -> NN
barks -> IN
